In [40]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

In [41]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [42]:
X_train[0]

array([1, 1, 1])

In [43]:
def entropy(y):
    entropy = 0
    if len(y)==0:
        return 0
    p = sum(y[y==1])/len(y)
    if (p==1 or p==0):
        return 0
    else:
        return -p*np.log2(p) - (1-p)*np.log2(1-p)

In [44]:
print(entropy(y_train))

1.0


In [45]:
def split_indices(X,node_indices,feature):
    left_indices = []
    right_indices = []
    for i in node_indices:
        if X[i][feature]==1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices,right_indices

In [46]:
split_indices(X_train,[0,1,2,3,4,5,6,7,8,9],0)

([0, 3, 4, 5, 7], [1, 2, 6, 8, 9])

In [47]:
def weighted_entropy(X_node,X_left,y_left,X_right,y_right):
    left_entropy = entropy(y_left)
    right_entropy = entropy(y_right)
    w_left = len(X_left) / len(X_node)
    w_right = len(X_right) / len(X_node)
    return w_left * left_entropy + w_right * right_entropy

In [48]:
def information_gain(X,y,node_indices,feature):
    left_indices,right_indices = split_indices(X_train,node_indices,feature)
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]

    information_gain = 0

    node_entropy = entropy(y_node)
    weight_entropy = weighted_entropy(X_node,X_left,y_left,X_right,y_right)
    information_gain = node_entropy - weight_entropy
    return information_gain


In [49]:
def get_best_split(X,y,node_indices):
    num_features = X.shape[1]

    best_feature = -1

    max_info_gain = 0
    for feature in range(num_features):
        info_gain = information_gain(X,y,node_indices,feature)
        if info_gain > max_info_gain : 
            max_info_gain = info_gain
            best_feature = feature
    return best_feature

In [50]:
def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth, tree):

    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   

    best_feature = get_best_split(X, y, node_indices) 
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    

    left_indices, right_indices = split_indices(X, node_indices, best_feature)
    tree.append((left_indices, right_indices, best_feature))
    
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1, tree)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1, tree)
    return tree

In [51]:
tree = []
build_tree_recursive(X_train,y_train,[0,1,2,3,4,5,6,7,8,9],"Root",max_depth=2,current_depth=0,tree=tree)

 Depth 0, Root: Split on feature: 0
- Depth 1, Left: Split on feature: 1
  -- Left leaf node with indices [0, 4, 5, 7]
  -- Right leaf node with indices [3]
- Depth 1, Right: Split on feature: 2
  -- Left leaf node with indices [1]
  -- Right leaf node with indices [2, 6, 8, 9]


[([0, 3, 4, 5, 7], [1, 2, 6, 8, 9], 0),
 ([0, 4, 5, 7], [3], 1),
 ([1], [2, 6, 8, 9], 2)]